
## Assignment 4: Simulate transactions and check double-spending prevention

In [ ]:
# ============================================
# ASSIGNMENT 4: Transaction & Double-Spending Prevention
# ============================================

import hashlib

class Wallet:
    def __init__(self, name, balance=100):
        self.name = name
        self.balance = balance
        # Simulate a public key for wallet identity
        self.public_key = hashlib.sha256(name.encode()).hexdigest()

    def display_info(self):
        print(f"{self.name} | Balance: {self.balance} coins | Key: {self.public_key[:8]}...")

class Blockchain:
    def __init__(self):
        self.transactions = []  # List of confirmed transactions
        self.pending_tx_ids = set()  # Track transaction IDs to prevent double-spending
    
    def create_transaction(self, sender, receiver, amount):
        """Create a unique transaction"""
        tx_data = f"{sender.public_key}{receiver.public_key}{amount}{len(self.transactions)}"
        tx_id = hashlib.sha256(tx_data.encode()).hexdigest()[:16]
        return {
            'tx_id': tx_id,
            'sender': sender.name,
            'sender_key': sender.public_key[:8],
            'receiver': receiver.name,
            'receiver_key': receiver.public_key[:8],
            'amount': amount
        }
    
    def process_transaction(self, tx, sender_wallet, receiver_wallet):
        """Process transaction with double-spending check"""
        print(f"\nProcessing Transaction: {tx['tx_id']}")
        print(f"  {tx['sender']} -> {tx['receiver']}: {tx['amount']} coins")
        
        # Check 1: Double-spending prevention (same transaction ID)
        if tx['tx_id'] in self.pending_tx_ids:
            print("  [REJECTED] Double-spending detected! Transaction already processed.")
            return False
        
        # Check 2: Sufficient balance
        if sender_wallet.balance < tx['amount']:
            print(f"  [REJECTED] Insufficient balance! Has: {sender_wallet.balance}, Needs: {tx['amount']}")
            return False
        
        # Process valid transaction
        sender_wallet.balance -= tx['amount']
        receiver_wallet.balance += tx['amount']
        self.transactions.append(tx)
        self.pending_tx_ids.add(tx['tx_id'])
        
        print("  [CONFIRMED] Transaction successful!")
        return True
    
    def show_ledger(self):
        print("\n" + "="*50)
        print("BLOCKCHAIN LEDGER")
        print("="*50)
        for i, tx in enumerate(self.transactions):
            print(f"{i+1}. [{tx['tx_id']}] {tx['sender']} -> {tx['receiver']}: {tx['amount']} coins")
        print("="*50)

# Initialize wallets
wallet_alice = Wallet("Alice", balance=100)
wallet_bob = Wallet("Bob", balance=100)

# Initialize blockchain
blockchain = Blockchain()

print("Blockchain initialized!")
print("\nInitial wallet states:")
wallet_alice.display_info()
wallet_bob.display_info()

Blockchain initialized!

Initial wallet states:
Alice | Balance: 100 coins | Key: 3bc51062...
Bob | Balance: 100 coins | Key: cd9fb1e1...


In [ ]:
# Simulate Transactions
print("\n" + "#"*50)
print("SIMULATING TRANSACTIONS")
print("-"*50)

# Transaction 1: Alice sends 30 coins to Bob (Valid)
tx1 = blockchain.create_transaction(wallet_alice, wallet_bob, 30)
blockchain.process_transaction(tx1, wallet_alice, wallet_bob)

# Transaction 2: Bob sends 20 coins to Alice (Valid)
tx2 = blockchain.create_transaction(wallet_bob, wallet_alice, 20)
blockchain.process_transaction(tx2, wallet_bob, wallet_alice)

# Show current balances
print("\n--- Current Balances ---")
print(f"Alice: {wallet_alice.balance} coins")
print(f"Bob: {wallet_bob.balance} coins")


##################################################
SIMULATING TRANSACTIONS
##################################################

Processing Transaction: 28f7989b196a3632
  Alice -> Bob: 30 coins
  [CONFIRMED] Transaction successful!

Processing Transaction: a81e4c547620ada0
  Bob -> Alice: 20 coins
  [CONFIRMED] Transaction successful!

--- Current Balances ---
Alice: 90 coins
Bob: 110 coins


In [ ]:
# Test Double-Spending Prevention
print("\n" + "#"*50)
print("DOUBLE-SPENDING PREVENTION TEST")
print("#"*50)

# Attempt 1: Try to replay the same transaction (tx1)
print("\nAttempt: Replay Transaction 1 (Double-Spend Attack)")
blockchain.process_transaction(tx1, wallet_alice, wallet_bob)

# Attempt 2: Try to spend more than balance
print("\nAttempt: Alice tries to send 500 coins (more than balance)")
tx_invalid = blockchain.create_transaction(wallet_alice, wallet_bob, 500)
blockchain.process_transaction(tx_invalid, wallet_alice, wallet_bob)

# Final balances (unchanged from double-spend attempts)
print("\n--- Final Balances (Unchanged) ---")
print(f"Alice: {wallet_alice.balance} coins")
print(f"Bob: {wallet_bob.balance} coins")


##################################################
DOUBLE-SPENDING PREVENTION TEST
##################################################

Attempt: Replay Transaction 1 (Double-Spend Attack)

Processing Transaction: 28f7989b196a3632
  Alice -> Bob: 30 coins
  [REJECTED] Double-spending detected! Transaction already processed.

Attempt: Alice tries to send 500 coins (more than balance)

Processing Transaction: 36ba06ac3e70cea4
  Alice -> Bob: 500 coins
  [REJECTED] Insufficient balance! Has: 90, Needs: 500

--- Final Balances (Unchanged) ---
Alice: 90 coins
Bob: 110 coins


In [ ]:
# Show complete ledger
blockchain.show_ledger()

# Final wallet info
print("\nFINAL WALLET STATES:")
wallet_alice.display_info()
wallet_bob.display_info()


BLOCKCHAIN LEDGER
1. [28f7989b196a3632] Alice -> Bob: 30 coins
2. [a81e4c547620ada0] Bob -> Alice: 20 coins

FINAL WALLET STATES:
Alice | Balance: 90 coins | Key: 3bc51062...
Bob | Balance: 110 coins | Key: cd9fb1e1...
